In [1]:
# Configuración inicial
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import os
import re
import time

# Configurar el modelo
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    temperature=0.7  # Balance entre creatividad y consistencia
)

print("✓ Modelo configurado para zero-shot prompting")

✓ Modelo configurado para zero-shot prompting


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# ── Configuración del modelo ──────────────────────────────
token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model = "openai/gpt-4.1"



llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1,
    streaming=True
)

# ── Configurar prompt con memoria ────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente profesional en el area automotriz, mas destinada al vulcanizado es decir parchado de neumaticos y montados de neumaticos, tambien tipo de llantas y radio, con gran conocimiento del trabajo."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm

# ── Almacén de historial por sesión ──────────────────────
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ── Envolver con memoria ──────────────────────────────────
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# ── Función de chat ───────────────────────────────────────
session_id = "mi_sesion"

def chat(mensaje):
    response = conversation.invoke(
        {"input": mensaje},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"Cliente: {mensaje}")
    print(f"Profesional Vulcanizador: {response.content}")
    print("-" * 40)

# ── Prueba — la IA debe recordar el contexto ─────────────
chat("Hola! Me llamo Jesus.")
chat("me gustaria saber que puedo hacer con un neumatico con los alambres salidos tambien tiene una forma de huevo deberia parchar o cambiar?")
chat("recuerdas mi nombre?")  # ← Esta prueba confirma que la memoria funciona

NameError: name 'model_name' is not defined

In [3]:
# ══════════════════════════════════════════
#  ZERO-SHOT PROMPTING - Vulcanizadora
# ══════════════════════════════════════════
from langchain_core.messages import HumanMessage

def chat_zero_shot(pregunta):
    """Responde sin ejemplos previos, solo con el rol definido."""
    
    prompt_zero_shot = f"""Eres un profesional experto en vulcanizado y neumáticos.
    
Tarea: Responde la consulta del cliente de forma clara y profesional.

Formato de respuesta:
- Diagnóstico: [qué tiene el neumático]
- Recomendación: [parchar o cambiar, con razón]
- Seguridad: [¿es urgente?]

Consulta del cliente: "{pregunta}"
"""
    
    response = llm.invoke([HumanMessage(content=prompt_zero_shot)])
    print(f"🙋 Cliente      : {pregunta}")
    print(f"🔧 Vulcanizador : {response.content}")
    print("-" * 50)

# Prueba
chat_zero_shot("Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?")
chat_zero_shot("¿Puedo seguir manejando con un clavo en el neumático?")

🙋 Cliente      : Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?
🔧 Vulcanizador : - Diagnóstico: El neumático presenta alambres salidos y deformación ("forma de huevo"), lo que indica daño estructural grave en la carcasa y en el área de rodado.  
- Recomendación: Debe cambiar el neumático inmediatamente. No es posible parchar ni reparar este tipo de daño, ya que la integridad del neumático está comprometida.  
- Seguridad: Es urgente. Continuar usando este neumático representa un alto riesgo de reventón y pérdida de control del vehículo. Reemplácelo lo antes posible.
--------------------------------------------------
🙋 Cliente      : ¿Puedo seguir manejando con un clavo en el neumático?
🔧 Vulcanizador : Diagnóstico: Su neumático tiene un clavo incrustado, lo que puede causar pérdida de presión de aire o daños internos, aunque no siempre es visible de inmediato.

Recomendación: Le recomiendo revisar el neumático lo antes posible. Si el clavo está en

In [8]:
# ══════════════════════════════════════════
#  FEW-SHOT PROMPTING - Vulcanizadora
# ══════════════════════════════════════════

def chat_few_shot(pregunta):
    """Responde aprendiendo el patrón de los ejemplos."""
    
    prompt_few_shot = f"""Eres un profesional experto en vulcanizado y neumáticos.
Responde siguiendo exactamente el estilo de estos ejemplos:

---
Cliente: "Tengo un clavo en la banda de rodadura del neumático"
Vulcanizador:
- Diagnóstico: Pinchazo en zona reparable (banda central)
- Recomendación: ✅ SE PUEDE PARCHAR - parche interno vulcanizado
- Seguridad: No es urgente, pero no demores más de 1 día

---
Cliente: "Mi neumático tiene una burbuja en el costado"
Vulcanizador:
- Diagnóstico: Hernia lateral, la carcasa interna está rota
- Recomendación: ❌ DEBE CAMBIARSE - no tiene reparación posible
- Seguridad: ⚠️ URGENTE, puede reventar al manejar

---
Cliente: "El neumático tiene un corte en el flanco de 3 cm"
Vulcanizador:
- Diagnóstico: Corte en zona no reparable (flanco lateral)
- Recomendación: ❌ DEBE CAMBIARSE - los flancos no se parchean
- Seguridad: ⚠️ URGENTE, riesgo de desinflado repentino

---
Cliente: "{pregunta}"
Vulcanizador:"""
    
    response = llm.invoke([HumanMessage(content=prompt_few_shot)])
    print(f"🙋 Cliente      : {pregunta}")
    print(f"🔧 Vulcanizador : {response.content}")
    print("-" * 50)

# Prueba
chat_few_shot("Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?")
chat_few_shot("¿Puedo seguir manejando con un clavo en el neumático?")

🙋 Cliente      : Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?
🔧 Vulcanizador : - Diagnóstico: Deformación estructural y alambres expuestos, daño severo en la carcasa
- Recomendación: ❌ DEBE CAMBIARSE - no es seguro repararlo
- Seguridad: ⚠️ URGENTE, riesgo alto de reventón mientras conduces
--------------------------------------------------
🙋 Cliente      : ¿Puedo seguir manejando con un clavo en el neumático?
🔧 Vulcanizador : - Diagnóstico: Pinchazo en zona reparable (banda de rodadura)
- Recomendación: ✅ SE PUEDE PARCHAR - parche interno vulcanizado
- Seguridad: ⚠️ NO RECOMENDADO, maneja con precaución y repara lo antes posible para evitar daños mayores.
--------------------------------------------------
